Step 1: Setup

In [11]:
!pip install -q duckdb

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
print("Connected. Ready to explore.")

Connected. Ready to explore.


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishuum13-star/flyrank-ml-internship-v3/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


Step 2: Section 1 — Signal Check #1 (Staleness)

In [12]:
import pandas as pd

In [13]:
# Signal Check 1: Staleness — does days-since-update relate to declining performance?
staleness = con.sql(f"""
    WITH feb AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_feb
        FROM '{BASE}/fact_content_daily_performance/month=2026-02/data_0.parquet'
        GROUP BY content_hash_id
    ),
    mar AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_mar
        FROM '{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet'
        GROUP BY content_hash_id
    )
    SELECT
        c.content_hash_id,
        DATE_DIFF('day', c.content_updated_date, DATE '2026-03-01') AS days_since_update,
        feb.impressions_feb, mar.impressions_mar
    FROM '{BASE}/dim_content.parquet' c
    JOIN feb ON c.content_hash_id = feb.content_hash_id
    JOIN mar ON c.content_hash_id = mar.content_hash_id
""").df()

staleness["declining"] = (staleness["impressions_mar"] < staleness["impressions_feb"]).astype(int)
staleness["staleness_bucket"] = pd.cut(
    staleness["days_since_update"],
    bins=[-1, 90, 180, 365, 100000],
    labels=["0-90d (fresh)", "91-180d", "181-365d", "365d+ (very stale)"]
)

bucket_table = staleness.groupby("staleness_bucket").agg(
    n=("declining", "size"),
    decline_rate=("declining", "mean")
).round(3)
print(bucket_table)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                        n  decline_rate
staleness_bucket                       
0-90d (fresh)       29905         0.399
91-180d              5026         0.118
181-365d             2173         0.041
365d+ (very stale)      0           NaN


/tmp/ipykernel_613/841295032.py:29: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_table = staleness.groupby("staleness_bucket").agg(


Step 3: Signal Check #2 (CTR vs Position)

In [14]:
ctr_check = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_clicks) AS clicks, SUM(gsc_impressions) AS impressions,
        AVG(gsc_avg_position) AS avg_position
    FROM '{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) >= 50
""").df()

ctr_check["ctr"] = ctr_check["clicks"] / ctr_check["impressions"]
ctr_check["position_bucket"] = pd.cut(ctr_check["avg_position"], bins=[0,3,10,20,100],
    labels=["1-3 (top)","4-10","11-20","21-100 (deep)"])

pos_table = ctr_check.groupby("position_bucket", observed=True).agg(n=("ctr","size"), avg_ctr=("ctr","mean")).round(4)
print(pos_table)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                     n  avg_ctr
position_bucket                
1-3 (top)         9686   0.0037
4-10             52218   0.0033
11-20            24294   0.0024
21-100 (deep)    29915   0.0013


Step 4: Text cell — Rule + Reason Codes
TOP-20 REVIEW (action: CTR_FIX for all, reason_code: CTR_FIX for all):

1. content_44f34c0a90047651 — 212K impressions, pos 7.35, CTR 0.011% vs median 0.168%.
   Highest-volume, highest-score case. Wrong if: this page's real intent is informational
   (e.g. a definition), where low CTR is normal regardless of position.

2. content_8e1334d6356668e3 — 135K impressions, pos 4.55 (strong position), CTR ~0%.
   Wrong if: it's a duplicate/paginated URL and users click a canonical variant instead.

3. content_fec55986a1868d62 — 124K impressions, pos 9.39, CTR ~0%. Wrong if: title/meta is
   broken or missing in the SERP — a technical issue, not a content one.

4. content_f6116743b00afc2d — 108K impressions, pos 9.54, CTR 0.014%. Wrong if: the keyword
   is mostly navigational (e.g. brand search) and users click a sitelink instead.

5. content_cd3d932d4e1c8db0 — 89K impressions, pos 7.79, CTR 0.0045%. Wrong if: search_volume
   is inflated by bot traffic, common on very generic queries.

6. content_7c6373141eae744a — 133K impressions, pos 5.79, CTR 0.063% (closest to median of
   the top 20). Wrong if: the bucket median is skewed by a few outlier high-CTR pages.

7. content_046fc480045b88f5 — 84K impressions, pos 7.29, CTR 0.0072%. Wrong if: a recent
   URL/redirect change means GSC hasn't caught up to true current performance.

8. content_9540d884af3e41fd — same client as #7, 82K impressions, pos 7.79, CTR 0.013%.
   Wrong if: #7 and #8 share one client-level tracking issue, not two separate content problems.

9. content_425715547c6a3ea8 — 72K impressions, pos 6.40, CTR 0.0042%. Wrong if: the keyword
   is heavily covered by an AI Overview/featured snippet, suppressing clicks regardless of position.

10. content_36fc1ee501ec072d — 73K impressions, pos 6.46, CTR 0.022%. Wrong if: this is a
    seasonal keyword and March is naturally a low-intent month for it.

11. content_4977e90c4d93cf9f — 74K impressions, pos 7.40, CTR 0.038%. Wrong if: content
    quality is already fine and the real problem is title/meta — a different fix than "improve content."

12. content_945d6ff91386c817 — 58K impressions, pos 6.15, CTR 0.0086%. Wrong if: this page
    underperforms specifically on mobile SERP layout, not overall.

13. content_37a6fac676c8cebb — 48K impressions, pos 4.41 (strong position), CTR 0.0083%.
    Wrong if: near-zero CTR at a strong position this consistently suggests a tracking or
    redirect bug, not a content problem — needs a manual SERP check first.

14. content_bf078007df823490 — 45K impressions, pos 7.91, CTR 0.0000% (literally zero
    clicks). Wrong if: zero clicks at this volume could mean a broken URL/error page, not
    a CTR problem — check status code before flagging.

15. content_1bb7d17cac7f6b78 — same client as #2, 55K impressions, pos 5.36, CTR 0.036%.
    Wrong if: this keyword cluster cannibalizes clicks from a stronger sibling page on the
    same client.

16. content_f57f0a707cf46a0a — 47K impressions, pos 6.80, CTR 0.019%. Wrong if: recently
    published and still in early SERP-ranking fluctuation, not yet stable.

17. content_bb2a9972810ddd72 — 54K impressions, pos 4.46 (strong position), CTR 0.041%.
    Wrong if: it's simply a lower-CTR content_type (e.g. image-heavy) rather than a fixable
    title/meta issue.

18. content_cf2264753938463b — same client as #16, 47K impressions, pos 4.11, CTR 0.023%.
    Wrong if: part of the same client-wide pattern as #16, better investigated at the
    client level than page-by-page.

19. content_39e19a3ec2d95f9d — 42K impressions, pos 9.10, CTR 0.0095%. Wrong if: genuinely
    low-intent informational query where low CTR is expected industry-wide.

20. content_0c5606abaaab3178 — 39K impressions, pos 5.69, CTR 0.0000% (zero clicks). Wrong
    if: same as #14 — zero clicks at this volume is suspicious enough to warrant a technical
    check before assuming a CTR/content problem.

SIGNAL CHECK 1 — Staleness (linked to refresh-flag logic): OPPOSITE. Fresh pages
(0-90 days) show 39.9% decline rate, older pages (181-365 days) show only 4.1% —
opposite of "stale pages decline more." Likely reflects new pages in a volatile
ramp-up phase vs. proven evergreen content. This signal does NOT support a
staleness rule.

SIGNAL CHECK 2 — CTR vs Position (linked to CTR-fix logic): CONFIRMED. Average CTR
drops steadily as position worsens — 0.37% at position 1-3, down to 0.13% at
position 21-100. Real, usable signal.

MY RULE (plain words): Flag a page for "CTR_FIX" if it ranks in positions 4-20
AND its CTR is below the median CTR for its own position bucket.

REASON CODES: CTR_FIX (underperforming CTR for position), HEALTHY (CTR at/above
median), LOW_VISIBILITY (impressions < 50, unreliable to score).

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [15]:
queue = con.sql(f"""
    SELECT content_hash_id, client_hash_id, SUM(gsc_clicks) AS clicks,
        SUM(gsc_impressions) AS impressions, AVG(gsc_avg_position) AS avg_position
    FROM '{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

queue["ctr"] = queue["clicks"] / queue["impressions"].replace(0, 1)
queue["position_bucket"] = pd.cut(queue["avg_position"], bins=[0,3,10,20,100],
    labels=["1-3 (top)","4-10","11-20","21-100 (deep)"])

reliable = queue[queue["impressions"] >= 50]
bucket_medians = reliable.groupby("position_bucket", observed=True)["ctr"].median()
queue["median_ctr_for_bucket"] = queue["position_bucket"].map(bucket_medians)

def assign_action(row):
    if row["impressions"] < 50:
        return "LOW_VISIBILITY", "CTR_FIX", 0.0
    if 4 <= row["avg_position"] <= 20 and row["ctr"] < row["median_ctr_for_bucket"]:
        gap = row["median_ctr_for_bucket"] - row["ctr"]
        return "CTR_FIX", "CTR_FIX", gap * row["impressions"]
    return "HEALTHY", "CTR_FIX", 0.0

results = queue.apply(assign_action, axis=1)
queue["action"] = results.apply(lambda x: x[0])
queue["reason_code"] = results.apply(lambda x: x[1])
queue["score"] = results.apply(lambda x: x[2])

ranked = queue.sort_values("score", ascending=False).reset_index(drop=True)
print("Action counts:", ranked["action"].value_counts().to_dict())
print(f"Top score: {ranked['score'].max():.2f}")
ranked.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Action counts: {'HEALTHY': 81115, 'LOW_VISIBILITY': 60624, 'CTR_FIX': 34999}
Top score: 332.98


,content_hash_id,client_hash_id,clicks,impressions,avg_position,ctr,position_bucket,median_ctr_for_bucket,action,reason_code,score
0,content_44f34c0a90047651,client_23a62021009f63c4,24.0,212404.0,7.346909,0.000113,4-10,0.001681,CTR_FIX,CTR_FIX,332.981513
1,content_8e1334d6356668e3,client_73cda7b4e4f265ea,1.0,134984.0,4.545582,0.000007,4-10,0.001681,CTR_FIX,CTR_FIX,225.863866
2,content_fec55986a1868d62,client_73cda7b4e4f265ea,1.0,124075.0,9.385150,0.000008,4-10,0.001681,CTR_FIX,CTR_FIX,207.529412
3,content_f6116743b00afc2d,client_62f4a7e64f5e0096,15.0,107584.0,9.536301,0.000139,4-10,0.001681,CTR_FIX,CTR_FIX,165.813445
4,content_cd3d932d4e1c8db0,client_9958f0a7ae1df715,4.0,89332.0,7.786219,0.000045,4-10,0.001681,CTR_FIX,CTR_FIX,146.137815
5,content_7c6373141eae744a,client_62f4a7e64f5e0096,83.0,132593.0,5.789019,0.000626,4-10,0.001681,CTR_FIX,CTR_FIX,139.845378
6,content_046fc480045b88f5,client_a80fca3f171ed1de,6.0,83788.0,7.289152,0.000072,4-10,0.001681,CTR_FIX,CTR_FIX,134.820168
7,content_9540d884af3e41fd,client_a80fca3f171ed1de,11.0,82376.0,7.794395,0.000134,4-10,0.001681,CTR_FIX,CTR_FIX,127.447059
8,content_425715547c6a3ea8,client_73cda7b4e4f265ea,3.0,71513.0,6.395691,0.000042,4-10,0.001681,CTR_FIX,CTR_FIX,117.189916
9,content_36fc1ee501ec072d,client_62f4a7e64f5e0096,16.0,73135.0,6.457292,0.000219,4-10,0.001681,CTR_FIX,CTR_FIX,106.915966


In [16]:
import os
os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Saved {len(ranked)} rows to work/outputs/baseline_action_score.csv")

Saved 176738 rows to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [17]:
# Backing code for the top-20 review above
top20_full = ranked.head(20)[["content_hash_id", "client_hash_id", "impressions",
                                "avg_position", "ctr", "action", "reason_code", "score"]]
top20_full


,content_hash_id,client_hash_id,impressions,avg_position,ctr,action,reason_code,score
0,content_44f34c0a90047651,client_23a62021009f63c4,212404.0,7.346909,0.000113,CTR_FIX,CTR_FIX,332.981513
1,content_8e1334d6356668e3,client_73cda7b4e4f265ea,134984.0,4.545582,0.000007,CTR_FIX,CTR_FIX,225.863866
2,content_fec55986a1868d62,client_73cda7b4e4f265ea,124075.0,9.385150,0.000008,CTR_FIX,CTR_FIX,207.529412
3,content_f6116743b00afc2d,client_62f4a7e64f5e0096,107584.0,9.536301,0.000139,CTR_FIX,CTR_FIX,165.813445
4,content_cd3d932d4e1c8db0,client_9958f0a7ae1df715,89332.0,7.786219,0.000045,CTR_FIX,CTR_FIX,146.137815
5,content_7c6373141eae744a,client_62f4a7e64f5e0096,132593.0,5.789019,0.000626,CTR_FIX,CTR_FIX,139.845378
6,content_046fc480045b88f5,client_a80fca3f171ed1de,83788.0,7.289152,0.000072,CTR_FIX,CTR_FIX,134.820168
7,content_9540d884af3e41fd,client_a80fca3f171ed1de,82376.0,7.794395,0.000134,CTR_FIX,CTR_FIX,127.447059
8,content_425715547c6a3ea8,client_73cda7b4e4f265ea,71513.0,6.395691,0.000042,CTR_FIX,CTR_FIX,117.189916
9,content_36fc1ee501ec072d,client_62f4a7e64f5e0096,73135.0,6.457292,0.000219,CTR_FIX,CTR_FIX,106.915966


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

WEAK PICKS: Looking at the top 20, one pattern stands out — heavy client concentration.
client_62f4a7e64f5e0096 appears 7 times (rows 3, 5, 9, 11, 12, 16, 19), and
client_73cda7b4e4f265ea appears 4 times (rows 1, 2, 8, 14). This is a weak pick pattern:
the rule is essentially flagging one client's entire content set repeatedly rather than
finding diverse opportunities. Rows #6/#7 (same client_a80fca3f171ed1de) and #16/#17 (same
client_e547b89c05043229) show the same issue at smaller scale. What would make these wrong:
if a client_hash_id has a site-wide technical problem (e.g. broken schema markup, slow page
speed) depressing CTR everywhere, this rule would flag every one of their pages individually
as separate "content" problems — masking one root cause as twenty different fixes. A better
version of this rule would first check the client-level average CTR gap before recommending
per-page fixes, to avoid recommending 7 near-identical actions for one client's systemic issue.

Rows #14 and #20 (content_bf078007df823490, content_0c5606abaaab3178) show literally 0.0000%
CTR at real volume (44K-39K impressions) — these are the weakest picks in the top 20, because
zero clicks at that volume is more consistent with a broken URL, tracking pixel, or redirect
loop than a "write better titles" content fix. These should be manually checked before action,
not auto-assigned CTR_FIX.

LEAKAGE CHECK: No future-window inputs — every feature (clicks, impressions, avg_position, ctr)
comes only from March 2026, the same month being scored; nothing from April-June is touched.
No label-derived inputs — the rule uses no target/outcome variable at all; it's a pure
threshold rule on observed CTR vs. its own position-bucket median, not a trained model with a
label. No product flags (e.g. an internal "needs_fix" column) were used — only

In [18]:
# Verify the client concentration claim with actual counts
top20_clients = ranked.head(20)["client_hash_id"].value_counts()
print("Client appearance counts in top 20:")
print(top20_clients)

# Verify zero-click rows
zero_click_rows = ranked.head(20)[ranked.head(20)["ctr"] == 0]
print(f"\nRows with 0% CTR in top 20: {len(zero_click_rows)}")
zero_click_rows[["content_hash_id", "impressions", "ctr"]]

Client appearance counts in top 20:
client_hash_id
client_62f4a7e64f5e0096    7
client_73cda7b4e4f265ea    4
client_23a62021009f63c4    2
client_a80fca3f171ed1de    2
client_e547b89c05043229    2
client_9958f0a7ae1df715    1
client_e5c2aa26a8598242    1
client_1a730cb2640a1abf    1
Name: count, dtype: int64

Rows with 0% CTR in top 20: 2


,content_hash_id,impressions,ctr
13,content_bf078007df823490,44707.0,0.0
19,content_0c5606abaaab3178,38865.0,0.0


## Self-check

Before you submit, confirm each line honestly:
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.